# 24. 경보 샷 설명 — 어느 변수가 튀었나 + 이상 원인 후보 (보고서 4장 작업자 지원)

`20a`(학습)·`20b`(테스트)가 저장한 모델과 점수를 그대로 사용 (다시 학습 안 함)

- **검사 순위**: 경보 샷을 점수 높은 순으로 → 이 순서대로 검사
- **상위 기여 변수**: Mahalanobis 점수²를 변수별로 분해 → 점수를 가장 많이 올린 변수 3개. 표기 = 이름(dev 값, 비율)
  - dev 값 = 같은 Run 직전 20샷 대비 몇 배 벗어났나 (+ 증가 / − 감소)
- **이상 원인 후보**: 사전에 정한 규칙(|dev| ≥ 3 기준)으로 붙인 해석 — 검증된 유형 분류기가 아님

| 후보 | 규칙 |
|---|---|
| 충전 이상(미성형형) | 충전 이상지표 ≤ −3, 또는 충전시간 ≤ −3 이고 사출속도 ≥ +3 |
| 쿠션 부족(재료 부족형) | 쿠션위치 ≤ −3 |
| 계량 지연형 | 계량시간 ≥ +3 |
| 흐름저항 상승(점도형) | 흐름저항 ≥ +3 또는 사출압 ≥ +3 |
| 열 상태 변화형 | 금형온도 또는 호퍼온도 \|dev\| ≥ 3 |

In [ ]:
import pandas as pd
from alarm_explain import alarm_table, hint_vs_actual, plot_contrib
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 20); pd.set_option('display.max_colwidth', 80)

## 1. 경보 목록 — q = 0.999 (정상 오경보율 0.1%)

In [ ]:
A = {}
for p in ['cn7', 'rg3']:
    A[p] = alarm_table(p, q=0.999)
    print(p.upper(), f'경보 {len(A[p])}샷')
    display(A[p].round(4))
    A[p].to_csv(f'./outputs/final_model/{p}_alarms_q0999.csv', index=False, encoding='utf-8-sig')

## 2. 원인 후보 × 실제 라벨 — q = 0.99 (경보를 넓혀서 경향 확인)
불량 경보에서 후보가 실제 유형(예: 미성형 ↔ 충전 이상)과 맞는지, 오경보는 어떤 후보로 나오는지

In [ ]:
for p in ['cn7', 'rg3']:
    print(p.upper()); display(hint_vs_actual(p, q=0.99))

## 3. 경보 샷 하나 자세히 보기 (CN7 검사 순위 1위)

In [ ]:
top = A['cn7'].iloc[0]['TimeStamp']
plot_contrib('cn7', top, save='./outputs/final_model/cn7_alarm_top1.png')

### 다른 샷 보기
`plot_contrib('cn7', '<TimeStamp>')` — 위 경보 목록의 TimeStamp를 넣으면 됨 (오경보 샷도 비교해 보기)